# 1. Instalasi dan Persiapan Library

In [ ]:
!pip install -q sentence-transformers rank_bm25 Sastrawi pandas numpy tqdm

import pandas as pd
import numpy as np
import re
import string
import torch
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from tqdm.auto import tqdm

# Setup Stopwords Indonesia
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Menggunakan Device: {device}")

# 2. Fungsi Preprocessing

In [ ]:
def preprocess_text(text):
    if pd.isna(text): return ""
    # Lowercase
    text = str(text).lower()
    # Hapus tanda baca dan karakter non-alfabet
    text = re.sub(r'[^a-z\s]', '', text)
    # Stopword Removal menggunakan Sastrawi
    text = stopword_remover.remove(text)
    # Hapus whitespace berlebih
    text = " ".join(text.split())
    return text

def get_tokens(text):
    return text.split()

# 3. Memuat Data dan Model Fine-Tuned

In [ ]:
# Load Dataset Pasangan Query-Tafsir 
df = pd.read_csv('data/processed/queries_synthetic_v2.csv') 

# Load Fine-Tuned Model SBERT
MODEL_PATH = 'models/sbert_tafsir_finetuned' 
model_sbert = SentenceTransformer(MODEL_PATH, device=device)

print(f"Memproses {len(df)} baris data...") 

# 4. Tahap Ekstraksi Fitur Semantik

In [ ]:
# Encoding secara batch 
queries = df['query'].tolist()
texts = df['tafsir_text'].tolist()

q_embs = model_sbert.encode(queries, batch_size=32, convert_to_tensor=True, show_progress_bar=True)
t_embs = model_sbert.encode(texts, batch_size=32, convert_to_tensor=True, show_progress_bar=True)

# Hitung Pairwise Similarity
cosine_scores = util.pairwise_cos_sim(q_embs, t_embs)
df['sbert_sim'] = cosine_scores.cpu().numpy()

# 5. Tahap Ekstraksi Fitur Leksikal (BM25, Jaccard, Overlap)

In [ ]:
# Preprocessing untuk Lexical Features & BM25
df['clean_query'] = df['query'].apply(preprocess_text)
df['clean_tafsir'] = df['tafsir_text'].apply(preprocess_text)

df['q_tokens'] = df['clean_query'].apply(get_tokens)
df['t_tokens'] = df['clean_tafsir'].apply(get_tokens)

# Hitung Overlap & Jaccard
def calc_lexical_metrics(row):
    set_q = set(row['q_tokens'])
    set_t = set(row['t_tokens'])
    if not set_q: return pd.Series([0.0, 0.0])
    
    intersect = len(set_q.intersection(set_t))
    union = len(set_q.union(set_t))
    
    overlap = intersect / len(set_q)
    jaccard = intersect / (union + 1e-9)
    return pd.Series([overlap, jaccard])

df[['overlap_score', 'jaccard_score']] = df.apply(calc_lexical_metrics, axis=1)

# Hitung BM25 Score
print("Menghitung BM25 Score...")
corpus_tokens = df['t_tokens'].tolist()
bm25_model = BM25Okapi(corpus_tokens)

bm25_scores = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    score = bm25_model.get_batch_scores(row['q_tokens'], [idx])[0]
    bm25_scores.append(score)

df['bm25_score'] = bm25_scores

# 6. Finalisasi dan Simpan file

In [ ]:
cols_to_keep = ['query', 'tafsir_text', 'sbert_sim', 'bm25_score', 'jaccard_score', 'overlap_score', 'label']
df_final = df[cols_to_keep]

OUTPUT_PATH = 'data/processed/dataset_training_final_FIXED.csv'
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"Dataset Final Berhasil Dibuat: {OUTPUT_PATH}")
print(df_final.head())